# VAW Phase 1 — Colab run: Stage D, what in the behavior features separates buildup clips from Normal?
Needs notebook 05 (context features) done. CPU, a few minutes. Method and how to read the result: `docs/ARCHITECTURE.md` section 10 and `docs/VERIFY_CONTEXT.md`.
This is the report's M4-M7 (features, baseline classifier, benchmark) done with controls: cross-validation **by clip**, every clip weighted equally, and a **shortcut probe**
that checks whether the way a clip was filmed (camera moving, brightness, resolution ...) explains the result instead of the behavior.

In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/KRITHIKSAI16/Violence-Against-Women.git"
if not os.path.isdir('/content/VAW'):
    !git clone -q {url} /content/VAW
else:
    !git -C /content/VAW pull -q {url} main
%cd /content/VAW
!pip install -q -r requirements.txt
!python -m pytest -q

## 1. Build the window table and run all experiments
Windows of 3 s per pair; for categories with a violent act only windows before the cut point are used. Prints: AUC for each feature set (style only / behavior / behavior + scene / all),
the same on static-camera clips only, per-category AUC against Normal, and which features matter. `--rebuild` recomputes the table (needed after notebook 05 is rerun).

In [ ]:
!python -m src.context.learn --config configs/colab.yaml --rebuild

## 2. How to read the output (write this down)
* **AUC 0.5 = chance, 1.0 = perfect.** Compare `style_only` with `behavior`. If style is as high as behavior, the dataset has a shortcut (e.g. the buildup categories are filmed with a
  moving camera, Normal mostly is not) and behavior results are unproven.
* **AUC static**: only clips with a still camera, the fairer comparison.
* **Per-category AUC vs Normal**: which categories look different from Normal at all.
* The labels are weak (the category, not what happens each second), so a modest AUC does not mean the detectors are wrong. The annotated benchmark (next stage) measures that.
* Compare with the **M9 rule gate** line: the learned model should beat it, otherwise the extra machinery has not paid off.

In [ ]:
import json
r = json.load(open('/content/drive/MyDrive/VAW_results/learn/results.json'))
for k in ('gb:style_only', 'gb:behavior', 'gb:behavior+scene', 'gb:all', 'logreg:behavior'):
    m = r['models'][k]
    print(f"{k:<20} all={m['all_clips']['auc']:.3f}  with-pair={m['clips_with_pair']['auc']:.3f}  static-cam={m['static_camera_clips']['auc']:.3f}")
print('rule gate:', r.get('rule_gate', {}).get('all_clips', {}).get('auc'))